# kl-divergence-gaussian-closed-form — worked example 1: KL divergence is zero when posterior equals the prior

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kl-divergence-gaussian-closed-form`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The closed-form KL divergence between a diagonal Gaussian `N(mu, sigma^2)` and the standard normal `N(0, I)` is `0.5 * sum(mu^2 + exp(2*logsigma) - 1 - 2*logsigma)`. When `mu=0` and `logsigma=0` (meaning `sigma=1`), the posterior exactly matches the prior and the KL is exactly 0. This is a useful correctness check.

## Worked solution

Step 1: The closed-form per-element KL is `0.5 * (mu^2 + exp(2*logsigma) - 1 - 2*logsigma)`. We can verify algebraically: plug in `mu=0, logsigma=0` to get `0.5 * (0 + 1 - 1 - 0) = 0`.

Step 2: Implement `kl_gaussian(mu, logsigma)` that computes per-element contributions, sums over the latent dimension, and returns (per_sample_kl, batch_mean_kl).

Step 3: Construct a batch where the first sample has `mu=0, logsigma=0` (posterior = prior), and verify its KL is zero.

Step 4: Construct a second sample with `mu=2, logsigma=1` (posterior far from prior) and verify its KL is large and positive.

In [ ]:
import torch as t

t.manual_seed(0)

def kl_gaussian(mu: t.Tensor, logsigma: t.Tensor):
    """Returns (per_sample_kl of shape (B,), scalar batch mean)."""
    # Form A: element-wise contribution
    per_elem = 0.5 * (mu.pow(2) + (2 * logsigma).exp() - 1 - 2 * logsigma)
    per_sample = per_elem.sum(dim=-1)   # sum over latent dim
    scalar = per_sample.mean()          # mean over batch
    return per_sample, scalar

# Batch of 3, latent_dim=4
# Sample 0: at prior (mu=0, logsigma=0) -> KL = 0
# Sample 1: shifted (mu=2, logsigma=0.5)
# Sample 2: compressed (mu=0, logsigma=-2)
mu = t.tensor([[0.0, 0.0, 0.0, 0.0],
               [2.0, 2.0, 2.0, 2.0],
               [0.0, 0.0, 0.0, 0.0]])
logsigma = t.tensor([[0.0, 0.0,  0.0, 0.0],
                     [0.5, 0.5,  0.5, 0.5],
                     [-2.0, -2.0, -2.0, -2.0]])

kl_per_sample, kl_mean = kl_gaussian(mu, logsigma)
print('per-sample KL:', kl_per_sample.tolist())
print('batch mean KL:', kl_mean.item())

assert abs(kl_per_sample[0].item()) < 1e-5, 'KL at prior should be 0'
assert kl_per_sample[1].item() > 0, 'shifted sample should have positive KL'
assert kl_per_sample[2].item() > 0, 'compressed posterior should have positive KL'
print('All checks passed.')